## Pattern ISC - Post by Post Analysis

Given that each subject watched posts in a different order, we analyze the data post by post.
This allows us to see how brain responses vary for each specific post across subjects, across post type.

**Analysis plan inspiration from:**
> Chen, J., Leong, Y., Honey, C. et al. Shared memories reveal shared structure in neural activity across individuals. Nat Neurosci 20, 115–125 (2017). https://doi.org/10.1038/nn.4450

**Analysis Steps:**
1. Denoised BOLD data is loaded from the previous notebook.
2. Post timings are loaded from the metadata - e-prime event files for each subject, for each run.
3. Post BOLD timeseries is extracted - then averaged across TRs within each post to have one value per post per voxel per subject.

(1) per-subject/run event loading → (2) TR windows → (3) event-wise pattern extraction with a 5×5×5 searchlight

### Extract post-level average BOLD patterns

In [ ]:
from pathlib import Path
import importlib
from yy_fmri_kit.event_isc import roi
importlib.reload(roi)
from yy_fmri_kit.event_isc.roi import run_all_subjects
from yy_fmri_kit.io.find_files import build_denoised_runs_dict

In [ ]:
DERIV = Path("/path/to/data/derivatives/denoised")

Build a dict of denoised runs files per subject

In [ ]:
runs_dict = build_denoised_runs_dict(
    derivatives_dir=DERIV,
    space="MNI152NLin2009cAsym",
    desc_keywords="nltoolsClean")

print(runs_dict)

In [ ]:
OUT    = Path("/path/to/data/derivatives/postbypost/roi")
EVENTS = Path("/path/to/behavioral_analyses/behavioral_data_fmri/combined_events_with_bids.csv")
MASK   = Path("/path/to/data/brain_masks/pieman_a1_2mm.nii")

Extract post-level average BOLD patterns per subject, per scan (and potentially timeshift)

In [ ]:
run_all_subjects(
    runs_dict=runs_dict,
    events_path=EVENTS,
    out_dir=OUT,
    subject_col="bids_id",
    run_col="run",
    shift_tr=4,
    mask_path=mask,
    onset_col="onset_s",
    duration_col="duration_s",
    time_unit="seconds")

Quick QC for the output:

In [ ]:
import numpy as np

out_dir = Path("/path/to/data/derivatives/postbypost/roi")
npz_files = list(out_dir.rglob("*_desc-roi_patterns.npz"))
npz_files[:3]
f = np.load(npz_files[0], allow_pickle=True)
f.files

post_ids = f["post_ids"]
data = f["data"]
print(post_ids.shape)
print(data.shape)



QC plots of extracted data matrices are shown below. Each row is a post, each column is a voxel (feature). Color indicates BOLD response level (denoised but not z-scored)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(data, aspect="auto", cmap="RdBu_r")
plt.colorbar()
plt.xlabel("features")
plt.ylabel("posts")
plt.show()


### Compute post-level ISC patterns across subjects for ROI

In [ ]:
import importlib
from yy_fmri_kit.event_isc import load_aligned
importlib.reload(load_aligned)
from yy_fmri_kit.event_isc.load_aligned import build_data_list_for_run

from yy_fmri_kit.isc import compute
importlib.reload(compute)
from yy_fmri_kit.isc.compute import compute_isc

Build data matrix: posts x features (voxels) per subject and compute ISC per post type

In [ ]:
run_type = "AntiLeft"

subjects, post_ids, data_list = build_data_list_for_run(OUT, run_type)

isc_subjectwise, isc_mean = compute_isc(
    data_list,
    return_subjectwise=True,
    fisher_z=True,      # recommended
    standardize="zscore"  # this z-scores across posts (your “time” axis)
)

print(run_type, "data shape:", data_list[0].shape)
print("isc_subjectwise:", isc_subjectwise.shape)  # (N, F)
print("isc_mean:", isc_mean.shape)                # (F,)


In [ ]:
np.save(f"isc_{run_type}_subjectwise.npy", isc_subjectwise)  # (N,F)
np.save(f"isc_{run_type}_mean.npy", isc_mean)                # (F,)

Compare real ISC top shuffeled ISC by post_id

In [ ]:
x = data_list[0].copy()
perm = np.random.permutation(x.shape[0])
data_list_shuf = data_list.copy()
data_list_shuf[0] = x[perm]

isc_subj_shuf, _ = compute_isc(data_list_shuf, return_subjectwise=True, fisher_z=True)
sub_isc_shuf = np.nanmean(isc_subj_shuf, axis=1)

print("Real mean ISC:", sub_isc.mean())
print("Shuffled mean ISC:", sub_isc_shuf.mean())


In [ ]:
plt.scatter(isc_subj_shuf, sub_isc_shuf)

Full pipeline example:

In [ ]:
from yy_fmri_kit.event_isc.extraction import ROIPatternExtractor
from yy_fmri_kit.static.event_isc.config import ExtractionConfig


In [ ]:
# ============================================
# STEP 1: BATCH EXTRACTION (all subjects)
# ============================================
config = ExtractionConfig(tr=1.5, shift_tr=4)
extractor = ROIPatternExtractor(config)

# Process ALL subjects at once (takes a while!)
summary = extractor.batch_extract(
    runs_dict=all_subjects_dict,  # ALL subjects
    events_df=all_events_df,       # ALL events
    output_dir=Path("./roi_output"),
    verbose=False  # Set True to see each post
)

print(f"Processed {len(all_subjects_dict)} subjects")
print(summary.groupby('status').size())

# ============================================
# STEP 2: BATCH ALIGNMENT (all run types)
# ============================================
aligner = PatternAligner(AlignmentConfig(strategy="intersection"))

# Align ALL subjects for ALL run types
aligned_results = aligner.align_all_runs(
    output_dir=Path("./roi_output"),
    run_types=["AntiLeft", "AntiRight", "ProLeft", "ProRight"],
)

# Save each aligned run type
for run_type, result in aligned_results.items():
    output_file = Path(f"./aligned/{run_type}_aligned.npz")
    aligner.save_aligned(result, output_file, run_type)
    
    print(f"{run_type}: {len(result['subjects'])} subjects, "
          f"{len(result['post_ids'])} posts")

# ============================================
# STEP 3: Load aligned data for ISC
# ============================================
import numpy as np

# Load one run type (all subjects aligned)
data = np.load("./aligned/AntiLeft_aligned.npz")

subjects = data["subjects"]   # Array of subject IDs
post_ids = data["post_ids"]   # Array of post IDs (canonical order)
patterns = data["data"]       # (n_subjects, n_posts, n_voxels)

print(f"Shape: {patterns.shape}")
# Example output: Shape: (15, 30, 1234)
# = 15 subjects, 30 posts, 1234 voxels

# Now ready for ISC computation!